# Machine Learning Practical Manual
## Data Preprocessing & Feature Engineering — All 8 Practicals
---

## PRACTICAL 1: Handling Missing Values
**Objective:** Detect missing values in a dataset and apply various imputation techniques including Mean, Median, Mode, SimpleImputer, and KNNImputer.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer

# ── STEP 1: Load any CSV dataset ──
# df = pd.read_csv("your_dataset.csv")

# Demo: Create a sample dataset with missing values
np.random.seed(42)
df = pd.DataFrame({
    'Age':    [25, np.nan, 30, 22, np.nan, 35, 28, np.nan, 40, 33],
    'Salary': [50000, 60000, np.nan, 45000, 55000, np.nan, 52000, 48000, np.nan, 61000],
    'Gender': ['Male', 'Female', np.nan, 'Male', 'Female', 'Male', np.nan, 'Female', 'Male', 'Female']
})

# ── STEP 2: Detect missing values ──
print("=" * 50)
print("Missing Values Count:")
print(df.isnull().sum())
print(f"Total Missing: {df.isnull().sum().sum()}")

# ── STEP 3: Separate numerical and categorical columns ──
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"\nNumerical columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")

# ── STEP 4: Mean Imputation ──
df_mean = df.copy()
df_mean[num_cols] = df_mean[num_cols].fillna(df_mean[num_cols].mean())
print("\nAfter Mean Imputation (missing):", df_mean.isnull().sum().sum())

# ── STEP 5: Median Imputation ──
df_median = df.copy()
df_median[num_cols] = df_median[num_cols].fillna(df_median[num_cols].median())
print("After Median Imputation (missing):", df_median.isnull().sum().sum())

# ── STEP 6: Mode Imputation (for categorical) ──
df_mode = df.copy()
df_mode[cat_cols] = df_mode[cat_cols].fillna(df_mode[cat_cols].mode().iloc[0])
print("After Mode Imputation (missing):", df_mode.isnull().sum().sum())

# ── STEP 7: SimpleImputer ──
si = SimpleImputer(strategy="mean")
df_simple = df.copy()
df_simple[num_cols] = si.fit_transform(df[num_cols])
print("After SimpleImputer (missing):", df_simple.isnull().sum().sum())

# ── STEP 8: KNNImputer ──
knn = KNNImputer(n_neighbors=5)
df_knn = df.copy()
df_knn[num_cols] = knn.fit_transform(df[num_cols])
print("After KNNImputer (missing):", df_knn.isnull().sum().sum())

# ── STEP 9: Verify ──
print("\n" + "=" * 50)
print("KNN Imputed DataFrame:")
print(df_knn)

---
## PRACTICAL 2: Data Normalization
**Objective:** Apply Min-Max Scaling and Standard Scaling (Z-score Normalization) on datasets and compare their effects on ML model performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris

# ── Load dataset ──
# df = pd.read_csv("your_dataset.csv")
# X = df.drop("target", axis=1).select_dtypes(include=[np.number])
# y = df["target"]

# Demo: Use Iris dataset
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── No Scaling (baseline) ──
model = KNeighborsClassifier()
model.fit(X_train, y_train)
acc_raw = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy (No Scaling):    {acc_raw:.4f}")

# ── Min-Max Scaling ──
scaler_mm = MinMaxScaler()
X_train_mm = scaler_mm.fit_transform(X_train)  # fit on train only
X_test_mm  = scaler_mm.transform(X_test)        # only transform test
model.fit(X_train_mm, y_train)
acc_mm = accuracy_score(y_test, model.predict(X_test_mm))
print(f"Accuracy (Min-Max):       {acc_mm:.4f}")

# ── Standard Scaling ──
scaler_ss = StandardScaler()
X_train_ss = scaler_ss.fit_transform(X_train)
X_test_ss  = scaler_ss.transform(X_test)
model.fit(X_train_ss, y_train)
acc_ss = accuracy_score(y_test, model.predict(X_test_ss))
print(f"Accuracy (Standard):      {acc_ss:.4f}")

print("\n" + "=" * 50)
print("Summary:")
print(f"  No Scaling  → {acc_raw:.4f}")
print(f"  Min-Max     → {acc_mm:.4f}")
print(f"  Standard    → {acc_ss:.4f}")

---
## PRACTICAL 3: Feature Scaling
**Objective:** Apply Log Scaling, Power Transform, and Robust Scaling on skewed datasets and compare their effects on data distribution and model performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing

# ── Load dataset ──
# df = pd.read_csv("your_dataset.csv")

# Demo: California Housing
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Check skewness
print("Skewness before transformation:")
print(X_train.skew().round(2))

# ── Log Scaling (for positive values only) ──
X_train_log = np.log1p(X_train.clip(lower=0))
X_test_log  = np.log1p(X_test.clip(lower=0))

# ── Power Transform (Yeo-Johnson — works for any value) ──
pt = PowerTransformer(method="yeo-johnson")
X_train_pt = pt.fit_transform(X_train)
X_test_pt  = pt.transform(X_test)

# ── Robust Scaling ──
rs = RobustScaler()
X_train_rs = rs.fit_transform(X_train)
X_test_rs  = rs.transform(X_test)

# ── Evaluate on all versions ──
print("\n" + "=" * 50)
print("RMSE Comparison:")
for name, Xtr, Xte in [("Raw",    X_train,    X_test),
                        ("Log",    X_train_log, X_test_log),
                        ("Power",  X_train_pt,  X_test_pt),
                        ("Robust", X_train_rs,  X_test_rs)]:
    lr = LinearRegression()
    lr.fit(Xtr, y_train)
    preds = lr.predict(Xte)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"  {name:<8} RMSE: {rmse:.4f}")

---
## PRACTICAL 4: Encoding Categorical Variables
**Objective:** Convert categorical text data into numerical format using One-Hot Encoding, Label Encoding, and Ordinal Encoding; and understand when to apply each technique.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# ── Load dataset ──
# df = pd.read_csv("your_dataset.csv")

# Demo: Create a sample dataset
df = pd.DataFrame({
    'Color':     ['Red', 'Blue', 'Green', 'Blue', 'Red', 'Green'],
    'Gender':    ['Male', 'Female', 'Male', 'Female', 'Male', 'Female'],
    'Size':      ['Small', 'Medium', 'Large', 'Small', 'Large', 'Medium'],
    'Education': ['High School', 'Bachelor', 'Master', 'PhD', 'Bachelor', 'Master'],
    'Price':     [100, 200, 150, 180, 120, 160]
})

cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print("Categorical columns:", cat_cols)
print("\nOriginal DataFrame:")
print(df)

# ── One-Hot Encoding ──
df_ohe = pd.get_dummies(df, columns=['Color', 'Gender'], drop_first=True)
print("\n" + "=" * 50)
print(f"Shape after OHE: {df_ohe.shape}")
print("After One-Hot Encoding:")
print(df_ohe.head(3))

# ── Label Encoding ──
df_le = df.copy()
le = LabelEncoder()
for col in ['Color', 'Gender']:
    df_le[col] = le.fit_transform(df_le[col].astype(str))
print("\nAfter Label Encoding:")
print(df_le[['Color', 'Gender']].head(3))

# ── Ordinal Encoding ──
df_ord = df.copy()

# Size: Small < Medium < Large
oe_size = OrdinalEncoder(categories=[['Small', 'Medium', 'Large']])
df_ord[['Size']] = oe_size.fit_transform(df_ord[['Size']])

# Education: High School < Bachelor < Master < PhD
oe_edu = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master', 'PhD']])
df_ord[['Education']] = oe_edu.fit_transform(df_ord[['Education']])

print("\nAfter Ordinal Encoding (Size & Education):")
print(df_ord[['Size', 'Education']])

---
## PRACTICAL 5: Feature Extraction
**Objective:** Extract meaningful numerical features from text data using TF-IDF, understand Word Embeddings (Word2Vec), and explain Image Feature Extraction using CNNs.

In [ ]:
# ── Part A: TF-IDF ──
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Demo: Text classification
# df = pd.read_csv("text_dataset.csv")  # needs "text" and "label" columns

# Demo dataset
texts = [
    "I love this movie", "This film is great", "Amazing and wonderful",
    "I hate this movie", "Terrible and boring film", "Worst movie ever",
    "Fantastic performance", "Great storyline", "Awful acting", "Poor direction"
]
labels = [1, 1, 1, 0, 0, 0, 1, 1, 0, 0]  # 1=Positive, 0=Negative

df_text = pd.DataFrame({'text': texts, 'label': labels})
X, y = df_text["text"], df_text["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# TF-IDF: Convert text to numerical features
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"TF-IDF Matrix Shape (train): {X_train_tfidf.shape}")
print(f"Top features: {tfidf.get_feature_names_out()[:10]}")

# Train and evaluate
lr = LogisticRegression(max_iter=500)
lr.fit(X_train_tfidf, y_train)
print(f"\nTF-IDF Accuracy: {accuracy_score(y_test, lr.predict(X_test_tfidf)):.4f}")

In [ ]:
# ── Part B: Word2Vec (Averaging embeddings) ──
# pip install gensim
try:
    from gensim.models import Word2Vec
    import numpy as np

    sentences = [text.lower().split() for text in texts]
    w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, epochs=10)

    def get_doc_vector(words, model):
        vectors = [model.wv[w] for w in words if w in model.wv]
        return np.mean(vectors, axis=0) if vectors else np.zeros(100)

    X_w2v = np.array([get_doc_vector(text.lower().split(), w2v_model) for text in texts])
    print("Word2Vec feature matrix shape:", X_w2v.shape)
    print("Sample vector (first 5 dims):", X_w2v[0][:5].round(4))
except ImportError:
    print("gensim not installed. Run: pip install gensim")

In [ ]:
# ── Part C: CNN Image Features (requires TensorFlow) ──
# pip install tensorflow
try:
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.preprocessing.image import load_img, img_to_array
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
    from tensorflow.keras.models import Model
    import numpy as np

    # Load pretrained CNN without top (classification) layer
    base_model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

    def extract_image_features(img_path):
        img = load_img(img_path, target_size=(224, 224))
        arr = preprocess_input(img_to_array(img))
        return base_model.predict(np.expand_dims(arr, 0))[0]

    print("MobileNetV2 loaded. Output feature vector size: 1280 dimensions")
    print("Usage: feature_vector = extract_image_features('image.jpg')")
    print("Feature vector shape: (1280,)")
except ImportError:
    print("TensorFlow not installed. Run: pip install tensorflow")
    print("CNN feature extraction requires TensorFlow/Keras.")

---
## PRACTICAL 6: Feature Selection
**Objective:** Identify the most important features using Correlation Analysis, Mutual Information, Recursive Feature Elimination (RFE), and Feature Importance methods.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_breast_cancer

# ── Load dataset ──
# df = pd.read_csv("your_dataset.csv")

# Demo: Breast Cancer dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 1. Correlation Analysis ──
corr = X_train.corrwith(pd.Series(y_train, name="target")).abs().sort_values(ascending=False)
top_corr_features = corr[corr > 0.3].index.tolist()
print("Top correlated features (|r| > 0.3):")
print(corr[corr > 0.3].round(3))

# Remove highly correlated feature pairs
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_cols = [col for col in upper.columns if any(upper[col] > 0.95)]
print(f"\nColumns to drop (highly correlated pairs): {drop_cols[:5]}")

# ── 2. Mutual Information ──
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)
top_mi = mi_df[mi_df > 0.01].index.tolist()
print("\nTop MI features:")
print(mi_df.head(10).round(4))

# ── 3. RFE ──
lr = LogisticRegression(max_iter=5000)
rfe = RFE(estimator=lr, n_features_to_select=10)
rfe.fit(X_train, y_train)
rfe_features = X_train.columns[rfe.support_].tolist()
print(f"\nRFE selected features: {rfe_features}")

# ── 4. Feature Importance (Random Forest) ──
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
fi = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
top_fi = fi[fi > 0.01].index.tolist()
print("\nTop RF Importance features:")
print(fi.head(10).round(4))

# ── Compare accuracy ──
print("\n" + "=" * 50)
print("Accuracy Comparison:")
for name, features in [("All Features", X_train.columns.tolist()),
                        ("Corr Features", top_corr_features),
                        ("RFE Features",  rfe_features),
                        ("RF Importance", top_fi)]:
    rf.fit(X_train[features], y_train)
    acc = accuracy_score(y_test, rf.predict(X_test[features]))
    print(f"  {name:<20} ({len(features):>2} features) → Accuracy: {acc:.4f}")

---
## PRACTICAL 7: Handling Imbalanced Data
**Objective:** Handle class imbalance using Oversampling, Undersampling, and SMOTE; and compare model performance before and after balancing.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.datasets import make_classification

# pip install imbalanced-learn
try:
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    from imblearn.under_sampling import RandomUnderSampler

    # ── Create imbalanced dataset ──
    X, y = make_classification(
        n_samples=1000, n_features=10, weights=[0.95, 0.05],
        random_state=42, n_informative=5
    )
    X = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(10)])

    print("Class distribution:", dict(zip(*np.unique(y, return_counts=True))))

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    def evaluate(name, X_tr, y_tr):
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_tr, y_tr)
        y_pred = rf.predict(X_test)
        f1 = f1_score(y_test, y_pred, average="weighted")
        print(f"\n{'='*50}")
        print(f"=== {name} ===")
        print(classification_report(y_test, y_pred))
        return f1

    # Baseline (no balancing)
    f1_raw = evaluate("Baseline (No Balancing)", X_train, y_train)

    # Random Oversampling
    ros = RandomOverSampler(random_state=42)
    X_ros, y_ros = ros.fit_resample(X_train, y_train)
    f1_ros = evaluate("Random Oversampling", X_ros, y_ros)

    # Random Undersampling
    rus = RandomUnderSampler(random_state=42)
    X_rus, y_rus = rus.fit_resample(X_train, y_train)
    f1_rus = evaluate("Random Undersampling", X_rus, y_rus)

    # SMOTE
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_sm, y_sm = smote.fit_resample(X_train, y_train)
    f1_sm = evaluate("SMOTE", X_sm, y_sm)

    print("\n" + "=" * 50)
    print("F1-Score Comparison:")
    print(f"  Baseline:           {f1_raw:.4f}")
    print(f"  OverSampling:       {f1_ros:.4f}")
    print(f"  UnderSampling:      {f1_rus:.4f}")
    print(f"  SMOTE:              {f1_sm:.4f}")

except ImportError:
    print("imbalanced-learn not installed. Run: pip install imbalanced-learn")

---
## PRACTICAL 8: Creating New Features (Feature Engineering)
**Objective:** Create new informative features using Feature Interaction, Polynomial Features, Date-Time Features, and Domain Knowledge-Based Features to improve model performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_breast_cancer

# ── Load dataset ──
# df = pd.read_csv("your_dataset.csv")

# Demo: Create sample dataset with domain-like columns
np.random.seed(42)
n = 200
df = pd.DataFrame({
    'income':  np.random.randint(20000, 120000, n),
    'debt':    np.random.randint(1000, 50000, n),
    'price':   np.random.randint(100000, 800000, n),
    'area':    np.random.randint(500, 5000, n),
    'weight':  np.random.randint(45, 120, n),
    'height':  np.random.randint(150, 200, n),
    'date':    pd.date_range(start='2023-01-01', periods=n, freq='D'),
    'target':  np.random.randint(0, 2, n)
})

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

# ── 1. Feature Interaction ──
df["price_per_area"]      = df["price"] / (df["area"] + 1e-5)
df["debt_ratio"]          = df["debt"]  / (df["income"] + 1e-5)
df["income_minus_debt"]   = df["income"] - df["debt"]
print("\nAfter Feature Interaction — New columns: price_per_area, debt_ratio, income_minus_debt")

# ── 2. Date-Time Features ──
df["hour"]       = df["date"].dt.hour
df["day_of_week"]= df["date"].dt.dayofweek
df["month"]      = df["date"].dt.month
df["year"]       = df["date"].dt.year
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["quarter"]    = df["date"].dt.quarter
df = df.drop("date", axis=1)
print("After DateTime Features — New: hour, day_of_week, month, year, is_weekend, quarter")

# ── 3. Domain Knowledge (BMI) ──
df["BMI"] = df["weight"] / ((df["height"] / 100) ** 2 + 1e-5)
df["BMI_category"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 25, 30, 60],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)
print("After Domain Features — New: BMI, BMI_category")
print("BMI distribution:\n", df["BMI_category"].value_counts())

# ── 4. Polynomial Features ──
num_cols = df.select_dtypes(include=[np.number]).columns.drop("target", errors="ignore")
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(df[num_cols])
print(f"\nOriginal features: {len(num_cols)} | After Polynomial (degree=2): {X_poly.shape[1]}")

# ── Compare accuracy before and after engineering ──
X = df.drop("target", axis=1).select_dtypes(include=[np.number])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
acc = accuracy_score(y_test, rf.predict(X_test))

print("\n" + "=" * 50)
print(f"Accuracy with engineered features: {acc:.4f}")

# Feature importances
fi = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\nTop 10 Feature Importances:")
print(fi.head(10).round(4))

---
## Final Summary & Best Practices

| Practical | Technique | When to Use | Best Method |
|-----------|-----------|-------------|-------------|
| P1: Missing | Mean/Median/KNN | Data has NaN | KNNImputer |
| P2: Normalize | MinMax/Standard | Distance/gradient algo | StandardScaler |
| P3: Scale | Log/Power/Robust | Skewed features | PowerTransform |
| P4: Encode | OHE/Label/Ordinal | Categorical data | OHE (nominal) |
| P5: Extract | TF-IDF/W2V/CNN | Text or Image data | Word2Vec/CNN |
| P6: Select | Corr/MI/RFE/RF | Many features | MI + RF Importance |
| P7: Imbalance | SMOTE/Over/Under | Skewed class ratio | SMOTE |
| P8: Engineer | Interaction/Poly/DT | Domain knowledge | Domain-based |

### ⚠️ Critical Rules:
1. **Always** fit scalers/imputers on **training data only**, then transform test data
2. **Never** apply SMOTE or resampling on test data
3. Use **F1-score / AUC-ROC** for imbalanced datasets, not accuracy
4. Use `drop_first=True` in `pd.get_dummies()` to avoid dummy variable trap
5. Split train/test **before** any preprocessing to prevent data leakage